# 🛗 Day 04a: LLD — Elevator System Design

---

## 🎯 What You'll Master Today
- Requirements & state machine
- Scheduling algorithms (SCAN, LOOK)
- Class design: Elevator, ElevatorSystem, Request
- Complete working implementation

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  ELEVATOR — STATE MACHINE                                        ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║                  ┌──────────┐                                    ║
║           ┌─────►│   IDLE   │◄─────┐                            ║
║           │      └────┬─────┘      │                            ║
║           │           │ request    │ queue empty                ║
║           │           ▼            │                            ║
║      no request ┌──────────┐  arrived                           ║
║           │     │ MOVING   │───────┘                            ║
║           │     │ UP/DOWN  │                                    ║
║           │     └────┬─────┘                                    ║
║           │          │ at floor                                  ║
║           │          ▼                                           ║
║           │     ┌──────────┐                                    ║
║           └─────│ STOPPED  │ (doors open, load/unload)          ║
║                 └──────────┘                                    ║
║                                                                   ║
║  Scheduling: SCAN (elevator algorithm)                           ║
║  • Move in one direction, serve all requests in that direction  ║
║  • When no more requests ahead, reverse direction               ║
║  • Like disk scheduling SCAN algorithm                          ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# 1. Enums and Request
# ============================================================

from enum import Enum
from collections import defaultdict

class Direction(Enum):
    UP = 1
    DOWN = -1
    IDLE = 0

class ElevatorState(Enum):
    IDLE = "IDLE"
    MOVING = "MOVING"
    STOPPED = "STOPPED"  # doors open

class Request:
    def __init__(self, floor: int, direction: Direction = None):
        self.floor = floor
        self.direction = direction  # external button press direction
    
    def __repr__(self):
        d = self.direction.name if self.direction else "internal"
        return f"Req(floor={self.floor}, {d})"

print("✅ Enums and Request defined")

In [ ]:
# ============================================================
# 2. Single Elevator (SCAN/LOOK Algorithm)
# ============================================================

class Elevator:
    def __init__(self, elevator_id: int, min_floor: int = 1, max_floor: int = 10):
        self.id = elevator_id
        self.current_floor = 1
        self.direction = Direction.IDLE
        self.state = ElevatorState.IDLE
        self.min_floor = min_floor
        self.max_floor = max_floor
        
        # Sets of floors to visit (for O(1) lookup)
        self.up_stops = set()    # floors to stop at going UP
        self.down_stops = set()  # floors to stop at going DOWN
    
    def add_request(self, floor: int, direction: Direction = None):
        """Add a floor to visit."""
        if floor > self.current_floor:
            self.up_stops.add(floor)
        elif floor < self.current_floor:
            self.down_stops.add(floor)
        # If at current floor, open doors immediately (handled in step)
        
        if self.direction == Direction.IDLE:
            if floor > self.current_floor:
                self.direction = Direction.UP
            elif floor < self.current_floor:
                self.direction = Direction.DOWN
    
    def step(self):
        """Simulate one time step (move one floor or open doors)."""
        if self.direction == Direction.IDLE:
            if self.up_stops:
                self.direction = Direction.UP
            elif self.down_stops:
                self.direction = Direction.DOWN
            else:
                return  # nothing to do
        
        # Check if current floor is a stop
        if self.direction == Direction.UP and self.current_floor in self.up_stops:
            self.up_stops.remove(self.current_floor)
            print(f"  🛗 Elevator {self.id}: STOP at floor {self.current_floor} (going UP)")
            return
        if self.direction == Direction.DOWN and self.current_floor in self.down_stops:
            self.down_stops.remove(self.current_floor)
            print(f"  🛗 Elevator {self.id}: STOP at floor {self.current_floor} (going DOWN)")
            return
        
        # Move
        if self.direction == Direction.UP:
            if self.up_stops:
                self.current_floor += 1
            else:
                # No more up stops → switch direction or idle
                if self.down_stops:
                    self.direction = Direction.DOWN
                    self.current_floor -= 1
                else:
                    self.direction = Direction.IDLE
        elif self.direction == Direction.DOWN:
            if self.down_stops:
                self.current_floor -= 1
            else:
                if self.up_stops:
                    self.direction = Direction.UP
                    self.current_floor += 1
                else:
                    self.direction = Direction.IDLE
    
    @property
    def is_idle(self):
        return not self.up_stops and not self.down_stops
    
    def __repr__(self):
        return f"Elevator({self.id}, floor={self.current_floor}, dir={self.direction.name})"

# Demo
e = Elevator(1, max_floor=10)
print(f"Start: {e}")

# Requests: go to floors 5, 3, 8, 2
for floor in [5, 3, 8, 2]:
    e.add_request(floor)
print(f"Up stops: {sorted(e.up_stops)}, Down stops: {sorted(e.down_stops)}")

print("\nSimulation:")
for _ in range(20):
    if e.is_idle:
        print(f"  🛗 Elevator {e.id}: IDLE at floor {e.current_floor}")
        break
    e.step()

In [ ]:
# ============================================================
# 3. Multi-Elevator System
# ============================================================

class ElevatorSystem:
    """
    Manages multiple elevators.
    Dispatches requests to the best elevator.
    """
    def __init__(self, num_elevators: int = 3, max_floor: int = 10):
        self.elevators = [Elevator(i+1, max_floor=max_floor) 
                         for i in range(num_elevators)]
    
    def _find_best_elevator(self, floor: int, direction: Direction) -> Elevator:
        """
        Pick the best elevator:
        1. Idle elevator closest to the floor
        2. Moving toward the floor in the same direction
        3. Any elevator (least busy)
        """
        best = None
        best_score = float('inf')
        
        for e in self.elevators:
            distance = abs(e.current_floor - floor)
            
            if e.direction == Direction.IDLE:
                score = distance  # idle + close = best
            elif (e.direction == Direction.UP and direction == Direction.UP 
                  and e.current_floor <= floor):
                score = distance  # going same way, will pass our floor
            elif (e.direction == Direction.DOWN and direction == Direction.DOWN 
                  and e.current_floor >= floor):
                score = distance
            else:
                score = distance + 10  # penalty for wrong direction
            
            if score < best_score:
                best_score = score
                best = e
        
        return best
    
    def request(self, floor: int, direction: Direction = None):
        """External hall button press."""
        elevator = self._find_best_elevator(floor, direction or Direction.UP)
        elevator.add_request(floor, direction)
        print(f"  📞 Floor {floor} ({direction.name if direction else 'call'}) → assigned to Elevator {elevator.id}")
    
    def step_all(self):
        for e in self.elevators:
            e.step()
    
    def status(self):
        for e in self.elevators:
            print(f"  {e}, up={sorted(e.up_stops)}, down={sorted(e.down_stops)}")

# Demo
system = ElevatorSystem(num_elevators=2, max_floor=10)
print("=== Multi-Elevator System ===")
system.request(5, Direction.UP)
system.request(3, Direction.DOWN)
system.request(8, Direction.UP)
print("\nStatus:")
system.status()

print("\nRunning steps...")
for i in range(15):
    all_idle = all(e.is_idle for e in system.elevators)
    if all_idle:
        print(f"  All elevators idle after {i} steps.")
        break
    system.step_all()

In [ ]:
# ============================================================
# 4. Design Discussion
# ============================================================

print("""
╔════════════════════════════════════════════════════════════════╗
║  ELEVATOR DESIGN — INTERVIEW DISCUSSION POINTS               ║
╠════════════════════════════════════════════════════════════════╣
║                                                                ║
║  Scheduling Algorithms:                                       ║
║  • FCFS: simple but inefficient (elevator zigzags)            ║
║  • SCAN: serve all in one direction, then reverse (our impl) ║
║  • LOOK: like SCAN but reverses when no more requests ahead  ║
║  • Destination Dispatch: group by destination (modern)        ║
║                                                                ║
║  Dispatch Strategy:                                           ║
║  • Nearest elevator (simple)                                  ║
║  • Same direction + nearest (better, what we implemented)    ║
║  • Zone-based: elevator 1 → floors 1-5, elevator 2 → 6-10   ║
║  • ML-based: predict traffic patterns (modern buildings)     ║
║                                                                ║
║  Extensions:                                                  ║
║  • VIP/Express elevator (skips floors)                        ║
║  • Weight capacity (don't pick up if full)                   ║
║  • Emergency mode (all go to ground floor)                   ║
║  • Maintenance mode (take elevator out of service)           ║
║  • Display: show direction + next stops on each floor        ║
║                                                                ║
║  Patterns used:                                               ║
║  • State pattern (IDLE/MOVING/STOPPED)                       ║
║  • Strategy pattern (scheduling algorithms)                   ║
║  • Observer pattern (floor displays subscribe to position)   ║
║  • Singleton (ElevatorSystem controller)                     ║
║                                                                ║
╚════════════════════════════════════════════════════════════════╝
""")

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Design elevator system | State machine + SCAN scheduling + dispatch to nearest same-direction |
| 2 | SCAN vs LOOK? | SCAN goes to max/min. LOOK reverses when no more requests ahead |
| 3 | How to dispatch? | Distance + direction match. Idle > same direction > opposite |
| 4 | Handle rush hour? | Zone-based assignment. Express elevators for high floors |
| 5 | Design patterns used? | State, Strategy, Observer, Singleton |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • State machine: IDLE → MOVING → STOPPED → ...          │
## │  • SCAN algorithm: one direction, serve all, reverse     │
## │  • Dispatch: nearest + same direction = best heuristic   │
## │  • Use sets for O(1) stop lookup                         │
## │  • Discuss extensions proactively                         │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Booking System Design (BookMyShow)**